In [198]:
from tensorflow import keras # type: ignore
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from datetime import datetime
import os
import pandas as pd
import numpy as np

In [199]:
#Opens a new dataframe with the Clean csv
cleancsv = pd.read_csv('CSV/CLEAN.csv')

In [200]:
#Convert data into Date time and create date filter
cleancsv['Date'] = pd.to_datetime(cleancsv['Date'])
cleancsv['Date'] = cleancsv['Date'] + pd.to_timedelta(cleancsv["Hr"], unit="h")
cleancsv.drop('Hr', axis=1, inplace=True)

"""
Use this in future if data set needs specific dates
prediction = data.loc{
    (untouched_csv['Date'] > datetime(x, x, x)) &
    (untouched_csv['Date'] < datetime(x, x, x,))
}
"""

"\nUse this in future if data set needs specific dates\nprediction = data.loc{\n    (untouched_csv['Date'] > datetime(x, x, x)) &\n    (untouched_csv['Date'] < datetime(x, x, x,))\n}\n"

In [201]:
#Create month colomn and restrict to only summer months
summer_mask = cleancsv["Date"].dt.month.isin([6, 7, 8])
cleancsv = cleancsv[summer_mask].reset_index(drop=True)

In [202]:
#Prepare colomns into variables
data_main_air_temp = cleancsv['Mainland Air Temp']
data_humidity_per = cleancsv['Humidity (%)']
data_wind_direction = cleancsv['Direction (A)']
data_wind_speed = cleancsv['Wind Speed (A)']
data_gusting = cleancsv['Gusting']
data_pressure = cleancsv['Atmospheric Pressure (IN)']
data_rainfall = cleancsv['Precipitation Rate']
data_bay_temp = cleancsv['Bay Temp']
data_salinity = cleancsv['Salinity']
data_lbi_temp = cleancsv['LBI Air Temp']
data_ocean_temp = cleancsv['Ocean Temp']
data_onshore_flag = cleancsv['Onshore']
data_upwelling_flag = cleancsv['upwelling_flag']

#saves all input data into one Numpy array
dataset = np.column_stack([
    data_main_air_temp.values,
    data_humidity_per.values,
    data_wind_direction.values,
    #data_wind_speed.values,
    data_gusting.values,
    data_pressure.values,
    data_rainfall.values,
    data_bay_temp.values,
    data_salinity.values,
    data_lbi_temp.values,
    data_ocean_temp.values,
    data_onshore_flag.values,
    data_upwelling_flag.values,
])

#Save output data into variables and reshape it to be a 2d array
output_data = data_wind_speed.values
output_data = np.array(output_data).reshape(-1, 1)

In [ ]:
#Length of training data
training_data_len = int(np.ceil(len(dataset) * 0.90)) #Use 90% of training data

In [ ]:
#Scaler
scaler_x= StandardScaler()
scaler_y= StandardScaler()

scaledx = scaler_x.fit_transform(dataset)
scaledy = scaler_y.fit_transform(output_data)

training_data_x = scaledx[:training_data_len] #90% of all data
training_data_y = output_data[:training_data_len] #90% of all data

X_train, y_train = [], []

In [205]:
#Sliding window over last 24 hrs
for i in range(24, training_data_len):
    X_train.append(training_data_x[i-24:i, :])
    y_train.append(training_data_y[i, 0])

#Convert lists to arrays
X_train = np.array(X_train)
y_train = np.array(y_train).reshape(-1, 1) #1D Array

In [206]:
test_x = scaledx[training_data_len-24:]
X_test = []

#rebuild window
for i in range(24, len(test_x)):
    X_test.append(test_x[i-24:i, :])

X_test = np.array(X_test)   # (samples_test, 24, n_features)
print(X_test.shape)


(109, 24, 12)


In [207]:
#Build the model
model = keras.models.Sequential()

In [208]:
#Layer Zero input_shape=(X_train.shape[1], 1)
model.add(keras.layers.Input(shape=(X_train.shape[1], X_train.shape[2])))

In [209]:
#First Layer input_shape=(X_train.shape[1], 1)
#model.add(keras.layers.LSTM(64, return_sequences=True))

In [210]:
#Second Layer
model.add(keras.layers.LSTM(32, return_sequences=False))

In [211]:
#3rd Layer (Dense)
model.add(keras.layers.Dense(32, activation="relu"))

In [212]:
#4th Layer (Dropout)
#model.add(keras.layers.Dropout(0.5))

In [213]:
#Final Output Layer (Dense)
model.add(keras.layers.Dense(1))

In [214]:
#Put all the layers together
model.summary()
model.compile(optimizer="adam",
              loss="mae",
              metrics=[keras.metrics.RootMeanSquaredError()])

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_16 (LSTM)                  │ (None, 32)             │         5,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,849 (26.75 KB)

 Trainable params: 6,849 (26.75 KB)

 Non-trainable params: 0 (0.00 B)

In [215]:
#Train the model

#epochs = # of runs
#batch size = how much data is in each batch
training = model.fit(
X_train,
    y_train,
    epochs=40,
    batch_size=32,
    validation_split=0.1,
    shuffle=True
)

Epoch 1/40
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 4.2389 - root_mean_squared_error: 5.0581 - val_loss: 4.4480 - val_root_mean_squared_error: 5.1859
Epoch 2/40
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.1736 - root_mean_squared_error: 5.0022 - val_loss: 4.3917 - val_root_mean_squared_error: 5.1363
Epoch 3/40
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.1172 - root_mean_squared_error: 4.9545 - val_loss: 4.3360 - val_root_mean_squared_error: 5.0873
Epoch 4/40
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.0611 - root_mean_squared_error: 4.9063 - val_loss: 4.2801 - val_root_mean_squared_error: 5.0384
Epoch 5/40
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.0055 - root_mean_squared_error: 4.8590 - val_loss: 4.2244 - val_root_mean_squared_error: 4.9898
Epoch 6/40
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3.9499 - root_mean_squared_error: 4.8117 - val_loss: 4.1688 - val_root_mean_squared_error: 4.9416
Epoch 7/40
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3.8947

In [216]:
prediction_scaled = model.predict(X_test)

# back to original units
prediction = scaler_y.inverse_transform(prediction_scaled)  


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step


In [217]:
# rows of the original dataframe that correspond to X_test / prediction
test_index_start = training_data_len
test_index_end = training_data_len + prediction.shape[0]

test_df = cleancsv.iloc[test_index_start:test_index_end].copy()

# add predicted column
test_df["wind_pred"] = prediction.ravel()

test_df.to_csv("CSV/predictions.csv", index=False)


In [218]:
# true wind speed for the same rows as predictions
y_test_true = cleancsv['Wind Speed (A)'].iloc[test_index_start:test_index_end].values

mae = mean_absolute_error(y_test_true, prediction.ravel())
print("Test MAE (m/s):", mae)

Test MAE (m/s): 6.058414355986709


In [219]:
print(prediction_scaled.shape)
print(prediction_scaled[:10].ravel())
print(prediction_scaled.mean(), prediction_scaled.std())

(109, 1)
[2.0959888 2.0959888 2.0959888 2.0959888 2.0959888 2.0959888 2.0959888
 2.0959888 2.0959888 2.0959888]
2.0959892 4.7683716e-07


In [220]:

training.history['val_loss']

[4.447967052459717,
 4.391733646392822,
 4.335996150970459,
 4.280148029327393,
 4.224375247955322,
 4.168807029724121,
 4.113099575042725,
 4.05776309967041,
 4.003017902374268,
 3.9499027729034424,
 3.8969056606292725,
 3.844308853149414,
 3.7926464080810547,
 3.7415049076080322,
 3.6920015811920166,
 3.6427719593048096,
 3.5943214893341064,
 3.5463297367095947,
 3.499554395675659,
 3.4520859718322754,
 3.405646562576294,
 3.360079050064087,
 3.3152225017547607,
 3.2702198028564453,
 3.226064682006836,
 3.183201789855957,
 3.1405117511749268,
 3.0986032485961914,
 3.0570971965789795,
 3.0192220211029053,
 2.9824326038360596,
 2.945833921432495,
 2.910006046295166,
 2.875244379043579,
 2.8408570289611816,
 2.8072867393493652,
 2.774883985519409,
 2.742568016052246,
 2.714008331298828,
 2.6837940216064453]

In [221]:
print("Wind speed mean/std:", output_data.mean(), output_data.std())
print("First 10 wind speeds:", output_data[:10].ravel())

Wind speed mean/std: 4.258974358974359 2.734957535493789
First 10 wind speeds: [3.8 4.  3.  2.7 2.  2.2 1.2 1.  1.  1.9]


In [222]:
i0 = training_data_len   # first test index
print("Example window X[0] (first test sample):")
print(cleancsv.iloc[i0-24:i0][["Wind Speed (A)", "Mainland Air Temp", "Direction (A)"]].head(10))
print("Target at i0:", cleancsv["Wind Speed (A)"].iloc[i0])

Example window X[0] (first test sample):
      Wind Speed (A)  Mainland Air Temp  Direction (A)
2051             2.5               24.7           15.0
2052             3.0               27.5           15.0
2053             3.9               28.9           14.0
2054             3.7               28.7           14.0
2055             5.9               27.3            9.0
2056             5.3               26.7            7.0
2057             4.7               26.7            9.0
2058             3.0               26.4           11.0
2059             1.8               25.2           12.0
2060             2.1               22.7           11.0
Target at i0: 8.5
